# Importing Library

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import SparseCategoricalAccuracy
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom , RandomTranslation , RandomContrast , RandomBrightness
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.regularizers import l2

# Importing Dataset

In [13]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 123
train_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="training",
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="validation",
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "data/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True
)

Found 6799 files belonging to 3 classes.
Using 5440 files for training.
Found 6799 files belonging to 3 classes.
Using 1359 files for validation.
Found 2278 files belonging to 3 classes.


# Data Preprocessing

In [14]:
data_augmentation = tf.keras.Sequential([
    RandomRotation(factor=(-0.025, 0.025)),
    RandomTranslation(height_factor=0.05, width_factor=0.05),
    RandomZoom(height_factor=(-0.05, 0.05), width_factor=(-0.05, 0.05)),
    RandomContrast(factor=0.1),
    RandomFlip(mode="horizontal"),
    RandomBrightness(factor=0.1)
], name="data_augmentation")

# Model Creation

In [15]:
def build_model(input_shape=IMG_SIZE + (3,), num_classes=3):
    base_model = ResNet50(
        include_top=False,
        input_shape=input_shape,
        weights="imagenet"
    )
    base_model.trainable = False 

    inputs = Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    outputs = Dense(num_classes, activation="softmax", kernel_regularizer=l2(1e-4))(x)

    model = Model(inputs, outputs, name="Human_Emotion_Detection_Model")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=[SparseCategoricalAccuracy(name="accuracy")]
    )
    return model, base_model

model, base_model = build_model()
model.summary()

Model: "Human_Emotion_Detection_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │         6,147 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,593,859 (90.00 MB)

 Trainable params: 6,147 (24.01 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

# Training The Model

In [16]:
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    ]
)

Epoch 1/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 22s 42ms/step - accuracy: 0.6206 - loss: 0.8535 - val_accuracy: 0.7116 - val_loss: 0.6721
Epoch 2/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.7272 - loss: 0.6535 - val_accuracy: 0.7476 - val_loss: 0.6206
Epoch 3/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 13s 39ms/step - accuracy: 0.7618 - loss: 0.5906 - val_accuracy: 0.7704 - val_loss: 0.6091
Epoch 4/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.7792 - loss: 0.5463 - val_accuracy: 0.7395 - val_loss: 0.6605
Epoch 5/5
340/340 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.8050 - loss: 0.5031 - val_accuracy: 0.7778 - val_loss: 0.5735


# Fine Tuning

## Total Layers of The Base Model

In [17]:
print("Base model has", len(base_model.layers), "layers.")

Base model has 175 layers.


## Tuning Process

In [18]:
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True


model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-6),
    loss="sparse_categorical_crossentropy",
    metrics=[SparseCategoricalAccuracy(name="accuracy")]
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        ModelCheckpoint("best_model.keras", save_best_only=True, monitor="val_loss"),
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)
    ]
)

Epoch 1/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 47s 83ms/step - accuracy: 0.7189 - loss: 0.6780 - val_accuracy: 0.7542 - val_loss: 0.5944 - learning_rate: 5.0000e-06
Epoch 2/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 21s 61ms/step - accuracy: 0.8447 - loss: 0.4009 - val_accuracy: 0.7756 - val_loss: 0.5475 - learning_rate: 5.0000e-06
Epoch 3/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 24s 70ms/step - accuracy: 0.9064 - loss: 0.2790 - val_accuracy: 0.7932 - val_loss: 0.5193 - learning_rate: 5.0000e-06
Epoch 4/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 21s 62ms/step - accuracy: 0.9447 - loss: 0.2032 - val_accuracy: 0.7984 - val_loss: 0.5117 - learning_rate: 5.0000e-06
Epoch 5/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 22s 66ms/step - accuracy: 0.9642 - loss: 0.1528 - val_accuracy: 0.8006 - val_loss: 0.5131 - learning_rate: 5.0000e-06
Epoch 6/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 22s 66ms/step - accuracy: 0.9744 - loss: 0.1202 - val_accuracy: 0.8006 - val_loss: 0.5202 - learning_rate: 5.0000e-06
Epoch 7/10
340/340 ━━━━━━━━━━━━━━━━━━━━ 20s 58ms/ste

# Evaluate

In [19]:
model.evaluate(test_ds)

143/143 ━━━━━━━━━━━━━━━━━━━━ 5s 35ms/step - accuracy: 0.8213 - loss: 0.5071


[0.5071170926094055, 0.8213344812393188]